In [ ]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')


Saving AdobeStock_85904537.jpeg to AdobeStock_85904537.jpeg
User uploaded file "AdobeStock_85904537.jpeg" with length 7666938 bytes


### Adding gradual noise to the image

This cell will take the image uploaded in the previous step, and gradually add noise to it over 40 steps. Each step's image will be saved to a folder named `noisy_images`.

In [ ]:
import numpy as np
from PIL import Image
import io
import os

# Check if an image was uploaded in the previous cell
if 'uploaded' in globals() and uploaded:
    # Get the first (and likely only) uploaded filename
    filename = list(uploaded.keys())[0]
    img_data = uploaded[filename]

    # Open the image using PIL from the bytes data
    try:
        original_image = Image.open(io.BytesIO(img_data))
        # Convert to RGB to handle different image modes consistently
        if original_image.mode != 'RGB':
            original_image = original_image.convert('RGB')
        original_array = np.array(original_image)
    except Exception as e:
        print(f"Error opening or converting image: {e}")
        print("Please ensure the uploaded file is a valid image.")
        exit()

    # Create a pure noise array of the same shape as the original image
    # This noise will be random values between 0-255
    pure_noise_array = np.random.randint(0, 256, original_array.shape, dtype=np.uint8)

    output_dir = "noisy_images"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    print(f"Saving noisy images to: {output_dir}/")

    for step in range(1, 41): # 40 steps, from 1 to 40
        alpha = step / 40.0 # Blending factor, goes from 0.025 to 1.0

        # Blend the original image with the pure noise image
        # Convert to float for calculation to avoid overflow, then back to uint8
        blended_array = ( (1 - alpha) * original_array.astype(np.float32) +
                          alpha * pure_noise_array.astype(np.float32) )

        # Clip values to ensure they are within the valid 0-255 range for image pixels
        blended_array = np.clip(blended_array, 0, 255).astype(np.uint8)

        # Convert back to PIL Image
        noisy_image = Image.fromarray(blended_array)

        # Save the image
        output_filename = os.path.join(output_dir, f"noisy_image_step_{step:02d}.png")
        noisy_image.save(output_filename)
        # print(f"Saved {output_filename}") # Uncomment to see each save message

    print(f"\nImage noise addition complete. {step} images saved in the '{output_dir}' directory.")
else:
    print("No image was uploaded. Please run the previous cell to upload an image first.")


Saving noisy images to: noisy_images/


### Creating a video from the noisy images

This cell will compile the sequence of noisy images into a video, with each image displayed for 0.25 seconds.

In [ ]:
import imageio.v2 as imageio
import os
from PIL import Image

output_dir = "noisy_images"
video_filename = "noisy_image_progression.mp4"

if not os.path.exists(output_dir):
    print(f"Error: Directory '{output_dir}' not found. Please run the previous cell first.")
else:
    image_files = []
    for i in range(1, 41): # Loop through steps 1 to 40
        filename = os.path.join(output_dir, f"noisy_image_step_{i:02d}.png")
        if os.path.exists(filename):
            image_files.append(filename)
        else:
            print(f"Warning: {filename} not found. Skipping.")

    if not image_files:
        print(f"No image files found in '{output_dir}'. Cannot create video.")
    else:
        # Ensure images are sorted correctly by step number
        image_files.sort()

        # Duration for each frame in seconds (1/4 of a second)
        frame_duration = 0.25

        # Create the video
        try:
            print(f"Creating video '{video_filename}' from {len(image_files)} images...")
            with imageio.get_writer(video_filename, mode='I', fps=1/frame_duration) as writer:
                for img_path in image_files:
                    image = imageio.imread(img_path)
                    writer.append_data(image)
            print(f"Video '{video_filename}' created successfully in the current directory.")
        except Exception as e:
            print(f"Error creating video: {e}")


Creating video 'noisy_image_progression.mp4' from 40 images...


Video 'noisy_image_progression.mp4' created successfully in the current directory.
